# Dove trovare i generatori

Generatore — a cosa associ questa parola? Forse ti fa pensare a un dispositivo elettronico, oppure a un macchinario pesante progettato per produrre energia elettrica o di altro tipo.

In Python, un **generatore** è una porzione di codice specializzata, capace di produrre una serie di valori e di controllare il processo di iterazione. Questo è il motivo per cui i generatori vengono molto spesso chiamati **iteratori** e, sebbene qualcuno possa trovarvi una distinzione davvero sottile, noi qui li tratteremo come un'unica entità.

Anche se potresti non rendertene conto, hai già incontrato i generatori moltissime volte in passato. Dai un'occhiata a questo semplicissimo frammento di codice:

```python
for i in range(5):
    print(i)

```

La funzione `range()` è, a tutti gli effetti, un generatore, il quale è (di fatto, nuovamente) un iteratore.

Qual è la differenza?

* **Una funzione** restituisce un singolo valore ben definito — che può essere il risultato di un calcolo più o meno complesso, come ad esempio un polinomio — e viene invocata una sola volta.
* **Un generatore** restituisce una serie di valori e, in generale, viene invocato (implicitamente) più di una volta.

Nell'esempio sopra, il generatore `range()` viene interpellato sei volte: fornisce cinque valori successivi da zero a quattro e, infine, segnala che la serie è completata.

Questo processo è del tutto trasparente per l'utente. Proviamo a fare luce sul meccanismo mostrando il funzionamento del **protocollo di iterazione** (*iterator protocol*).

Il protocollo di iterazione definisce il modo in cui un oggetto deve comportarsi per conformarsi alle regole imposte dal contesto delle istruzioni `for` e `in`. Un oggetto che rispetta questo protocollo è chiamato **iteratore**.

Un iteratore deve obbligatoriamente fornire due metodi:

* `__iter__()`: deve restituire l'oggetto stesso e viene invocato una sola volta (è necessario a Python per avviare correttamente l'iterazione);
* `__next__()`: ha lo scopo di restituire il valore successivo (il primo, il secondo, e così via) della serie desiderata. Viene invocato automaticamente dai costrutti `for`/`in` ad ogni nuova iterazione; se non ci sono più valori da restituire, il metodo deve sollevare l'eccezione `StopIteration`.

Ti sembra strano? Niente affatto. Dai un'occhiata all'esempio nell'editor per vedere come implementarlo.

In [2]:
class Fib:
    def __init__(self, nn):
        print("__init__")
        self.__n = nn
        self.__i = 0
        self.__p1 = self.__p2 = 1

    def __iter__(self):
        print("__iter__")
        return self

    def __next__(self):
        print("__next__")		
        self.__i += 1
        if self.__i > self.__n:
            raise StopIteration
        if self.__i in [1, 2]:
            return 1
        ret = self.__p1 + self.__p2
        self.__p1, self.__p2 = self.__p2, ret
        return ret


for i in Fib(10):
    print(i)
    

__init__
__iter__
__next__
1
__next__
1
__next__
2
__next__
3
__next__
5
__next__
8
__next__
13
__next__
21
__next__
34
__next__
55
__next__


Abbiamo costruito una classe in grado di iterare attraverso i primi $n$ valori (dove $n$ è un parametro del costruttore) della successione di Fibonacci.

Ti ricordiamo che i numeri di Fibonacci ($Fib$) sono definiti come segue:


$$Fib_1 = 1$$

$$Fib_2 = 1$$

$$Fib_i = Fib_{i-1} + Fib_{i-2}$$

In altre parole:

* I primi due numeri di Fibonacci sono uguali a 1;
* Qualsiasi altro numero di Fibonacci è la somma dei due precedenti (es. $Fib_3 = 2$, $Fib_4 = 3$, $Fib_5 = 5$, e così via).

Analizziamo il codice nel dettaglio:

* **Righe da 2 a 6:** il costruttore della classe stampa un messaggio (lo useremo per tracciare il comportamento della classe) e prepara alcune variabili (`__n` per memorizzare il limite della serie, `__i` per tenere traccia del numero corrente di Fibonacci da restituire, e `__p1` insieme a `__p2` per salvare i due numeri precedenti);
* **Righe da 8 a 10:** il metodo `__iter__` ha l'obbligo di restituire l'oggetto iteratore stesso. Il suo scopo potrebbe sembrare un po' ambiguo in questo contesto, ma non c'è alcun mistero: prova a immaginare un oggetto che non è un iteratore in sé (ad esempio, una collezione di elementi), ma uno dei suoi componenti interni lo è ed è in grado di scansionare la collezione; il metodo `__iter__` dovrebbe estrarre tale iteratore e affidargli l'esecuzione del protocollo. Come puoi vedere, nel nostro caso il metodo inizia la sua azione stampando un messaggio;
* **Righe da 12 a 21:** il metodo `__next__` è il responsabile della creazione della sequenza. È un po' prolisso, ma questo dovrebbe renderlo più leggibile: per prima cosa stampa un messaggio, poi aggiorna il conteggio dei valori generati e, se raggiunge la fine della sequenza, interrompe l'iterazione sollevando l'eccezione `StopIteration`. Il resto del codice è semplice e riflette fedelmente la definizione matematica che abbiamo visto poco fa;
* **Righe 24 e 25:** mettono finalmente all'opera l'iteratore all'interno di un ciclo.

Osserva bene cosa succede:

* L'oggetto iteratore viene istanziato per primo;
* Successivamente, Python invoca il metodo `__iter__` per accedere all'iteratore effettivo;
* Il metodo `__next__` viene invocato undici volte: le prime dieci producono i valori utili della sequenza, mentre l'undicesima interrompe l'iterazione.

L'esempio precedente mostra una soluzione in cui l'oggetto iteratore fa parte di una classe più complessa. Il codice non è particolarmente sofisticato, ma presenta il concetto in modo molto chiaro.

Dai un'occhiata al codice nell'editor per vedere come questa struttura può essere organizzata.

In [3]:
class Fib:
    def __init__(self, nn):
        self.__n = nn
        self.__i = 0
        self.__p1 = self.__p2 = 1

    def __iter__(self):
        print("Fib iter")
        return self

    def __next__(self):
        self.__i += 1
        if self.__i > self.__n:
            raise StopIteration
        if self.__i in [1, 2]:
            return 1
        ret = self.__p1 + self.__p2
        self.__p1, self.__p2 = self.__p2, ret
        return ret


class Class:
    def __init__(self, n):
        self.__iter = Fib(n)

    def __iter__(self):
        print("Class iter")
        return self.__iter


object = Class(8)

for i in object:
    print(i)

Class iter
1
1
2
3
5
8
13
21


Abbiamo integrato l'iteratore `Fib` all'interno di un'altra classe (possiamo dire che l'abbiamo composto dentro la classe `Class`). Esso viene istanziato insieme all'oggetto stesso di `Class`.

L'oggetto della classe può essere utilizzato come un iteratore quando (e solo quando) risponde positivamente all'invocazione del metodo `__iter__`. Questa classe è in grado di farlo e, se viene interpellata in questo modo, fornisce un oggetto capace di rispettare il protocollo di iterazione.

Questo è il motivo per cui l'output del codice è identico a quello precedente, sebbene l'oggetto della classe `Fib` non venga utilizzato esplicitamente all'interno del contesto del ciclo `for`.

# L'istruzione yield

Il protocollo di iterazione non è particolarmente difficile da capire e da usare, ma è indiscutibile che sia piuttosto scomodo da implementare.

Il disagio principale deriva dalla necessità di dover salvare lo stato dell'iterazione tra le successive invocazioni di `__next__()`. Nell'esempio precedente, l'iteratore `Fib` era costretto a memorizzare con precisione il punto in carezza si era interrotta l'ultima chiamata (ovvero il numero calcolato e i valori dei due elementi precedenti). Questo rende il codice più lungo, pesante e meno leggibile.

È per questo motivo che Python offre un modo molto più efficace, pratico ed elegante per scrivere gli iteratori. Questo concetto si basa fondamentalmente su un meccanismo specifico e potente introdotto dalla parola chiave `yield`.

Puoi pensare alla parola chiave `yield` come a una sorella più intelligente dell'istruzione `return`, ma con una differenza essenziale. Prendi come esempio questa funzione:

In [4]:
def fun(n):
    for i in range(n):
        return i

Ha un aspetto bizzarro, non trovi? È evidente che il ciclo `for` non ha alcuna possibilità di completare nemmeno la sua prima esecuzione, poiché il `return` lo interromperà irrevocabilmente alla prima iterazione. Inoltre, invocare nuovamente la funzione non cambierà il risultato: il ciclo `for` ripartirà da zero e si interromperà subito dopo.

Possiamo dire che una funzione del genere non è in grado di salvare e ripristinare il proprio stato tra le invocazioni successive. Di conseguenza, non può essere utilizzata come generatore.

Ora proviamo a sostituire una sola parola nel codice:

In [5]:
def fun(n):
    for i in range(n):
        yield i

Abbiamo inserito `yield` al posto di `return`. Questa piccola modifica trasforma la funzione in un **generatore**, e l'esecuzione dell'istruzione `yield` produce effetti davvero interessanti.

Prima di tutto, fornisce il valore dell'espressione specificata subito dopo la parola chiave (proprio come farebbe `return`), ma **senza perdere lo stato della funzione**. I valori di tutte le variabili vengono "congelati" e restano in attesa dell'invocazione successiva, momento in cui l'esecuzione riprenderà esattamente da dove si era interrotta (e non da capo, come accadrebbe dopo un `return`).

C'è un limite importante da tenere a mente: una funzione di questo tipo non deve essere invocata esplicitamente come se fosse una funzione normale, perché di fatto non lo è più; si tratta a tutti gli effetti di un **oggetto generatore** (*generator object*). Se la richiami direttamente, otterrai l'identificatore dell'oggetto e non la serie di valori che ti aspetti.

Per lo stesso motivo, la funzione precedente (quella con l'istruzione `return`) può essere invocata solo in modo esplicito e non può fungere da generatore.

# Come costruire un generatore

Mettiamo subito all'opera il nuovo generatore all'interno di un ciclo:

In [6]:
def fun(n):
    for i in range(n):
        yield i
 
for v in fun(5):
    print(v)

0
1
2
3
4


Riesci a indovinare l'output? Verranno stampati in colonna i numeri da `0` a `4`.

Cosa succede se hai bisogno di un generatore che produca le prime $n$ potenze di 2? Niente di più semplice. Guarda il codice qui sotto:

In [7]:
def powers_of_2(n):
    power = 1
    for i in range(n):
        yield power
        power *= 2
 
for v in powers_of_2(8):
    print(v)

1
2
4
8
16
32
64
128


In questo caso, l'output mostrerà la sequenza geometrica delle potenze (1, 2, 4, 8, 16, 32, 64, 128). Copia il codice nell'editor e digitalo per verificare le tue intuizioni.

## List comprehension

I generatori possono essere impiegati direttamente anche all'interno delle *list comprehension*, come in questo esempio:

In [8]:
def powers_of_2(n):
    power = 1
    for i in range(n):
        yield power
        power *= 2
 
t = [x for x in powers_of_2(5)]
print(t)

[1, 2, 4, 8, 16]


Eseguendo questo codice, otterrai una lista vera e propria: `[1, 2, 4, 8, 16]`.

## La funzione list()

La funzione integrata `list()` può convertire una serie di elementi successivi generati da un generatore in una lista reale, occupandosi di consumare l'iterazione:

In [9]:
def powers_of_2(n):
    power = 1
    for i in range(n):
        yield power
        power *= 2
 
t = list(powers_of_2(3))
print(t)

[1, 2, 4]


Prova a prevedere l'output (che sarà `[1, 2, 4]`) ed esegui il codice per controllare.

## L'operatore in

Inoltre, il contesto logico creato dall'operatore `in` permette di sfruttare un generatore per effettuare controlli di appartenenza al volo:

In [10]:
def powers_of_2(n):
    power = 1
    for i in range(n):
        yield power
        power *= 2
 
for i in range(20):
    if i in powers_of_2(4):
        print(i)

1
2
4
8


Cosa stamperà il programma? Poiché `powers_of_2(4)` genera i valori 1, 2, 4 e 8, il ciclo verificherà quali numeri compresi tra 0 e 19 fanno parte di questa serie, stampando a schermo proprio `1`, `2`, `4` e `8`.


## Il generatore della successione di Fibonacci

Vediamo ora come implementare il generatore per i numeri di Fibonacci. Noterai subito come questa versione appaia decisamente più snella e pulita rispetto alla controparte orientata agli oggetti basata sul protocollo di iterazione diretto:

In [11]:
def fibonacci(n):
    p = pp = 1
    for i in range(n):
        if i in [0, 1]:
            yield 1
        else:
            n_val = p + pp
            pp, p = p, n_val
            yield n_val
 
fibs = list(fibonacci(10))
print(fibs)

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


*(Nota: abbiamo rinominato leggermente la variabile interna in `n_val` per evitare confusione con il parametro `n` della funzione).*

L'output generato sarà la lista dei primi 10 numeri di Fibonacci: `[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]`.


## Qualcosa in più sulle list comprehension

Dovresti già ricordare le regole che governano la creazione e l'uso di quel costrutto speciale di Python chiamato **list comprehension** — un modo semplice, compatto e di grande impatto visivo per generare liste e popolarle con dei contenuti.

Nel caso avessi bisogno di rispolverare rapidamente l'argomento, nell'editor troverai un breve promemoria pronto all'uso.

In [12]:
list_1 = []

for ex in range(6):
    list_1.append(10 ** ex)

list_2 = [10 ** ex for ex in range(6)]

print(list_1)
print(list_2)

[1, 10, 100, 1000, 10000, 100000]
[1, 10, 100, 1000, 10000, 100000]


Ci sono due parti all'interno del codice nell'editor, ed entrambe creano una lista contenente le prime potenze naturali di dieci.

La prima parte impiega il metodo classico e routinario utilizzando un ciclo `for` esteso, mentre la seconda sfrutta la *list comprehension* e costruisce la lista direttamente *in situ*, senza la necessità di scrivere un blocco di codice separato.

A prima vista potrebbe sembrare che la lista venga creata "dentro se stessa", ma ovviamente non è così: dietro le quinte Python deve eseguire quasi le stesse identiche operazioni del primo frammento. È indiscutibile, tuttavia, che questo secondo formalismo sia decisamente più elegante e permetta a chi legge il codice di evitare dettagli superflui.

# L'espressione condizionale (Operatore Ternario)

C'è una sintassi molto interessante che vale la pena esaminare adesso. La sua utilità non è limitata solo alle *list comprehension*, ma bisogna ammettere che le comprehension rappresentano l'ambiente ideale in cui applicarla.

Si tratta dell'**espressione condizionale** — un costrutto (spesso chiamato operatore ternario) che permette di selezionare uno tra due valori differenti in base al risultato di un'espressione booleana.

La struttura è la seguente:

```python
espressione_uno if condizione else espressione_due

```

Può sembrare un po' insolita a prima vista, ma è importante tenere a mente che **non si tratta di un'istruzione condizionale** (come un classico blocco `if-else` strutturato). In realtà, non è affatto un'istruzione: è un **operatore**.

Il valore restituito da questo operatore sarà pari a `espressione_uno` quando la `condizione` è `True`, e a `espressione_due` in caso contrario.

Un buon esempio pratico ti aiuterà a fare chiarezza. Dai un'occhiata al codice nell'editor per vedere come viene integrato all'interno di una list comprehension.

In [13]:
the_list = []

for x in range(10):
    the_list.append(1 if x % 2 == 0 else 0)

print(the_list)

[1, 0, 1, 0, 1, 0, 1, 0, 1, 0]


Il codice popola una lista con valori pari a `1` e `0`: se l'indice di un particolare elemento è dispari, l'elemento viene impostato a `0`, altrimenti viene impostato a `1`.

Semplice? Forse non a prima vista. Elegante? Indiscutibilmente.

Si può usare lo stesso trucco all'interno di una *list comprehension*? Certamente sì, ed è proprio qui che questo operatore esprime al meglio la sua potenza e compattezza.

Dai un'occhiata all'esempio nell'editor per vedere come si combinano perfettamente questi due costrutti.

In [14]:
the_list = [1 if x % 2 == 0 else 0 for x in range(10)]

print(the_list)

[1, 0, 1, 0, 1, 0, 1, 0, 1, 0]


Compattezza ed eleganza: queste sono le due parole che vengono subito in mente guardando il codice.

Quindi, cosa hanno in comune i generatori e le *list comprehension*? Esiste un legame tra loro? Sì, un legame piuttosto sottile, ma inequivocabile.

Basta un singolo, piccolo cambiamento sintattico per trasformare una *list comprehension* in un generatore.

## List comprehension vs. Generatori

Ora guarda attentamente il codice qui sotto e prova a individuare il dettaglio che trasforma una *list comprehension* in un generatore:

```python
# Soluzione A: List Comprehension
lista = [x for x in range(5)]

# Soluzione B: Espressione Generatrice (Generator Expression)
generatore = (x for x in range(5))

```

La differenza sta tutta nelle **parentesi**:

* Le parentesi quadre `[...]` dicono a Python di creare e popolare immediatamente una **lista** in memoria;
* Le parentesi tonde `(...)` creano invece una **generator expression** (espressione generatrice).

In questo secondo caso, Python non alloca spazio per tutti gli elementi e non esegue il ciclo subito. Al contrario, restituisce un oggetto generatore pronto a produrre i valori uno alla volta, solo quando gli verranno esplicitamente richiesti (ad esempio in un ciclo `for` o tramite la funzione `next()`), garantendo un enorme risparmio di memoria quando si lavora con grandi quantità di dati.

In [15]:
the_list = [1 if x % 2 == 0 else 0 for x in range(10)]
the_generator = (1 if x % 2 == 0 else 0 for x in range(10))

for v in the_list:
    print(v, end=" ")
print()

for v in the_generator:
    print(v, end=" ")
print()

1 0 1 0 1 0 1 0 1 0 
1 0 1 0 1 0 1 0 1 0 


Esatto, la differenza sta proprio nelle parentesi! Le parentesi quadre creano una comprehension, mentre le parentesi tonde creano un generatore.

Come facciamo ad essere assolutamente certi che la seconda assegnazione crei un generatore e non una lista? La prova definitiva si ottiene applicando la funzione integrata `len()` a entrambe le entità:

* `len(la_lista)` restituirà `10`. Un comportamento chiaro, ovvio e prevedibile, poiché tutti gli elementi esistono già contemporaneamente in memoria.
* `len(il_generatore)` solleverà immediatamente un'eccezione, mostrando il seguente messaggio di errore:

```text
Output
TypeError: object of type 'generator' has no len()

```

Questo accade perché un generatore non conosce a priori la quantità totale di elementi che produrrà: non contiene dati stoccati in memoria, ma soltanto la *regola* (o il codice) per generarli un po' alla volta su richiesta.

Naturalmente, non è sempre obbligatorio salvare la lista o il generatore all'interno di una variabile. Puoi crearli direttamente *on the fly*, proprio nel punto esatto in cui ne hai bisogno — come in questo esempio:

In [16]:
for v in [1 if x % 2 == 0 else 0 for x in range(10)]:
    print(v, end=" ")
print()
 
for v in (1 if x % 2 == 0 else 0 for x in range(10)):
    print(v, end=" ")
print()

1 0 1 0 1 0 1 0 1 0 
1 0 1 0 1 0 1 0 1 0 


> **Nota bene:** Anche se l'output a schermo appare identico, i due cicli lavorano sotto il cofano in modi completamente diversi. Nel primo ciclo, la lista viene creata e allocata in memoria interamente nella sua totalità prima ancora che l'iterazione abbia inizio. Nel secondo ciclo, la lista non esiste affatto: ci sono solo valori successivi prodotti dal generatore, uno alla volta, su richiesta. Ti invito a fare i tuoi esperimenti per toccare con mano la differenza nelle performance e nell'uso della memoria.

# La funzione lambda (Funzioni Anonime)

Il concetto di **funzione lambda** è preso in prestito dalla matematica, più nello specifico da quella branca chiamata *Calcolo Lambda*, anche se le due nozioni non sono esattamente la stessa cosa.

Mentre i matematici usano il calcolo lambda in sistemi formali legati alla logica, alla ricorsione o alla dimostrazione di teoremi, noi programmatori usiamo le funzioni lambda principalmente per semplificare il codice, renderlo più asciutto e immediato da leggere.

In breve, una funzione lambda è una **funzione senza nome** (proprio per questo viene definita anche *funzione anonima*). Un'affermazione del genere fa sorgere spontanea una domanda: come si fa a usare qualcosa che non ha un identificativo per essere richiamato?

Fortunatamente non è un problema: se ne hai davvero la necessità puoi comunque assegnare una lambda a una variabile (dandole di fatto un nome), ma in moltissimi contesti la funzione lambda esiste e lavora rimanendo totalmente in incognito.

La dichiarazione di una funzione lambda non somiglia affatto a quella di una funzione standard con il costrutto `def`. La sua struttura essenziale è questa:

```python
lambda parametri: espressione

```

Questo costrutto valuta l'espressione utilizzando i parametri passati in input e ne **restituisce automaticamente il risultato** (senza bisogno di scrivere esplicitamente la parola chiave `return`).


Come sempre, un esempio pratico ci aiuterà a fare chiarezza. Nel frammento di codice che segue definiamo tre funzioni lambda, assegnandole temporaneamente a dei nomi per vedere come interagirci. Guardalo con attenzione:


In [17]:
two = lambda: 2
sqr = lambda x: x * x
pwr = lambda x, y: x ** y
 
for a in range(-2, 3):
    print(sqr(a), end=" ")
    print(pwr(a, two()))

4 4
1 1
0 0
1 1
4 4


Analizziamo insieme come funzionano queste tre istruzioni:

* **La prima lambda** è una funzione anonima priva di parametri (*parameterless*) che restituisce sempre il valore fisso `2`. Dal momento che l'abbiamo assegnata a una variabile di nome `two`, la funzione ha perso il suo stato di anonimato e possiamo usare questo nome per invocarla normalmente, ad esempio tramite `two()`.
* **La seconda lambda** è una funzione anonima a un singolo parametro che calcola e restituisce il valore del suo argomento elevato al quadrato. Anche in questo caso l'abbiamo memorizzata in una variabile per potervi fare riferimento in modo esplicito.
* **La terza lambda** accetta invece due parametri e restituisce il valore del primo sollevato alla potenza del secondo. Il nome della variabile scelto per ospitare questa lambda parla da sé. Abbiamo evitato di utilizzare l'identificativo `pow` proprio per non generare sovrapposizioni o confusioni con la funzione integrata di Python che ha lo stesso nome e lo stesso scopo.

Questo esempio è sufficientemente chiaro per mostrare come vengono dichiarate le lambda e come si comportano, ma non dice nulla sul perché siano necessarie e su quale sia il loro scopo, dato che potrebbero essere tutte sostituite con le classiche funzioni di Python.
Qual è il vantaggio reale?

## Come si usano le lambda e a cosa servono?

L'aspetto più interessante delle funzioni lambda è che puoi usarle nella loro forma pura — ovvero come blocchi di codice anonimi progettati esclusivamente per calcolare un risultato al volo.

Immagina di aver bisogno di una funzione (che chiameremo `print_function`) che stampi i valori di una determinata (altra) funzione per una serie di argomenti selezionati.
Vogliamo che `print_function` sia universale: deve accettare come argomenti sia un insieme di valori inseriti in una lista, sia la funzione stessa da valutare, senza che nulla sia scritto in modo rigido (*hardcoded*) nel codice.

Dai un'occhiata all'esempio nell'editor per vedere come abbiamo implementato questa idea.

In [18]:
def print_function(args, fun):
    for x in args:
        print('f(', x,')=', fun(x), sep='')


def poly(x):
    return 2 * x**2 - 4 * x + 2


print_function([x for x in range(-2, 3)], poly)

f(-2)=18
f(-1)=8
f(0)=2
f(1)=0
f(2)=2


Analizziamo il codice nel dettaglio. La funzione `print_function()` accetta due parametri:

* **Il primo** è una lista di argomenti di cui vogliamo stampare i risultati;
* **Il secondo** è una funzione che deve essere invocata tante volte quanti sono i valori raccolti all'interno del primo parametro.

> **Nota:** abbiamo definito anche una funzione chiamata `poly()` — questa è la funzione di cui andremo a stampare i valori. Il calcolo che esegue non è particolarmente sofisticato, si tratta di un polinomio (da cui il nome) nella forma:
> 
> $$f(x) = 2x^2 - 4x + 2$$
> 
> 

Il nome della funzione viene poi passato a `print_function()` insieme a un insieme di cinque argomenti diversi, generati direttamente tramite una *list comprehension*.

Possiamo evitare di definire la funzione `poly()` se non intendiamo usarla più di una volta? Certamente sì — ed è proprio questo il grande vantaggio che una funzione lambda porta con sé.

Guarda l'esempio qui sotto. Riesci a notare la differenza?

```python
def print_function(args, fun):
    for x in args:
        print('f(', x, ')=', fun(x), sep='')

print_function([x for x in range(-2, 3)], lambda x: 2 * x**2 - 4 * x + 2)

```

La funzione `print_function()` è rimasta esattamente la stessa, ma la funzione `poly()` è scomparsa. Non ne abbiamo più bisogno, perché il polinomio si trova ora direttamente all'interno dell'invocazione di `print_function()`, sotto forma di una lambda definita in questo modo:

```python
lambda x: 2 * x**2 - 4 * x + 2

```

Il codice è diventato più corto, più pulito e decisamente più leggibile.


Adesso ti mostriamo un altro contesto in cui le lambda si rivelano estremamente utili. Iniziamo con la descrizione di `map()`, una funzione integrata di Python. Il suo nome non è molto descrittivo, ma l'idea alla base è semplice e la funzione in sé è davvero utilissima.

## Le lambda e la funzione `map()`

Nel caso più semplice possibile, la funzione `map()`:

```python
map(funzione, lista)

```

accetta due argomenti:

1. Una funzione;
2. Una lista.

Questa descrizione è in realtà estremamente semplificata, poiché:

* Il secondo argomento di `map()` può essere una qualsiasi entità iterabile (ad esempio una tupla, o un generatore stesso);
* `map()` può accettare anche più di due argomenti.

La funzione `map()` applica la funzione passata come primo argomento a tutti gli elementi del secondo argomento, e **restituisce un iteratore** che fornisce uno alla volta i risultati di queste elaborazioni.

Puoi usare l'iteratore risultante direttamente in un ciclo, oppure convertirlo in una lista vera e propria usando la funzione `list()`.

Riesci a intravedere come una funzione lambda possa incastrarsi alla perfezione in questo meccanismo?

Dai un'occhiata al codice nell'editor per vedere un esempio in cui abbiamo combinato `map()` con due diverse funzioni lambda.

In [19]:
list_1 = [x for x in range(5)]
list_2 = list(map(lambda x: 2 ** x, list_1))
print(list_2)

for x in map(lambda x: x * x, list_2):
    print(x, end=' ')
print()

[1, 2, 4, 8, 16]
1 4 16 64 256 


Ecco spiegato l'intreccio del codice:

* Per prima cosa, viene costruita la `list_1` con i valori interi da `0` a `4`;
* Successivamente, viene usata la funzione `map()` insieme alla prima lambda per creare una nuova lista, in cui ogni elemento corrisponde a `2` elevato alla potenza del rispettivo valore preso da `list_1`;
* A questo punto, la `list_2` viene stampata a schermo;
* Nel passaggio seguente, si applica nuovamente la funzione `map()` per sfruttare direttamente l'iteratore restituito e stampare al volo tutti i valori che produce. Come puoi notare, qui abbiamo messo in campo la seconda lambda, che si occupa semplicemente di elevare al quadrato ogni elemento proveniente da `list_2`.

Prova a immaginare lo stesso identico codice scritto senza l'uso delle lambda. Sarebbe migliore o più chiaro? Decisamente no.

## Le lambda e la funzione `filter()`

Un'altra funzione nativa di Python che può essere notevolmente impreziosita e snellita dall'applicazione di una lambda è `filter()`.

Questo costrutto si aspetta lo stesso identico tipo di argomenti di `map()`, ma svolge un compito completamente diverso: filtra gli elementi del secondo argomento basandosi sulle direttive che riceve dalla funzione specificata come primo parametro (la quale viene invocata per ogni singolo elemento, esattamente come accade in `map()`).

Gli elementi che, passati alla funzione, restituiscono un valore logico `True` superano il filtro, mentre tutti gli altri vengono scartati e rifiutati.

L'esempio nell'editor mostra la funzione `filter()` in azione per aiutarti a comprenderne visivamente il comportamento.

In [26]:
from random import seed, randint

seed(0)
data = [randint(-10,10) for x in range(5)]
filtered = list(filter(lambda x: x > 0 and x % 2 == 0, data))

print(data)
print(filtered)

[2, 3, -9, -2, 6]
[2, 6]


> **Nota:** Abbiamo fatto uso del modulo `random` per inizializzare il generatore di numeri casuali (da non confondere con i generatori di valori di cui abbiamo appena parlato) tramite la funzione `seed()`, e per produrre cinque valori interi casuali compresi tra -10 e 10 utilizzando la funzione `randint()`.
> La lista viene poi filtrata e vengono accettati solo i numeri che sono contemporaneamente **maggiori di zero** e **pari**.

Naturalmente è improbabile che tu ottenga gli stessi identici numeri esguendo il codice, ma questo è l'aspetto che avevano i nostri risultati:

```text
[2, 6]

```

*(Nota: i numeri esatti dipendono dal valore di `seed` utilizzato nel programma, ma in ogni caso la lista finale conterrà esclusivamente numeri positivi e divisibili per 2).*

### Uno sguardo rapido alle closure

Partiamo da una definizione: una **closure** (o chiusura) è una tecnica di programmazione che permette di memorizzare e conservare l'accesso a determinati valori anche quando il contesto (lo scope) in cui sono stati originariamente creati non esiste più.

Suona complicato? Un pochino sì. Analizziamo prima un esempio errato per capire il problema:

```python
def outer(par):
    loc = par

var = 1
outer(var)
 
print(par)  # Errore!
print(loc)  # Errore!

```

Questo codice fallisce in modo evidente. Le ultime due righe solleveranno un'eccezione di tipo `NameError`: né `par` né `loc` sono accessibili all'esterno della funzione. Entrambe le variabili nascono, vivono e muoiono esclusivamente durante il tempo di esecuzione di `outer()`.

Ora guarda il codice nell'editor. Abbiamo modificato la struttura in modo significativo per mostrare come nasce una closure funzionante.

In [27]:
def outer(par):
    loc = par

    def inner():
        return loc
    return inner


var = 1
fun = outer(var)
print(fun())
    

1


Nel codice è presente un elemento completamente nuovo: una funzione (chiamata `inner`) definita all'interno di un'altra funzione (chiamata `outer`).

Come funziona? Esattamente come qualsiasi altra funzione, con l'unica eccezione che `inner()` può essere invocata solo dall'interno di `outer()`. Possiamo dire che `inner()` è uno strumento privato di `outer()` — nessun'altra parte del codice esterno può accedervi direttamente.

Analizziamo i passaggi con attenzione:

* La funzione `inner()` restituisce il valore della variabile accessibile all'interno del suo scope; questo è possibile perché `inner()` ha accesso a tutte le entità a disposizione di `outer()`.
* La funzione `outer()` restituisce la funzione `inner()` stessa. Più precisamente, restituisce un riferimento a quella specifica istanza di `inner()` che è stata "congelata" al momento dell'invocazione di `outer()`. Questa funzione preserva intatto tutto il suo ambiente circostante (*lexical environment*), incluso lo stato di tutte le variabili locali. Ciò significa che il valore di `loc` viene conservato con successo, nonostante l'esecuzione di `outer()` sia terminata da tempo.

Una closure deve essere invocata rispettando esattamente il modo in cui è stata dichiarata. Nell'esempio qui sotto:

```python
def outer(par):
    loc = par
 
    def inner():
        return loc
    return inner
 
var = 1
fun = outer(var)
print(fun())

```

La funzione `inner()` è priva di parametri, di conseguenza dobbiamo invocarla senza passare alcun argomento (usando le parentesi vuote `fun()`).

Ora guarda il codice nell'editor. È assolutamente possibile dichiarare una closure dotata di un numero arbitrario di parametri — ad esempio uno solo, esattamente come accade nella funzione `power()`.

In [28]:
def make_closure(par):
    loc = par

    def power(p):
        return p ** loc
    return power


fsqr = make_closure(2)
fcub = make_closure(3)

for i in range(5):
    print(i, fsqr(i), fcub(i))

0 0 0
1 1 1
2 4 8
3 9 27
4 16 64


Questo significa che la closure non solo si serve dell'ambiente congelato per funzionare, ma può anche modificarne il comportamento in modo dinamico sfruttando i valori che le vengono passati da fuori al momento dell'invocazione.

Questo scenario evidenzia una circostanza ancora più interessante: puoi generare tutte le closure che desideri partendo da un unico, identico frammento di codice sorgente. Nel nostro caso, lo facciamo tramite una funzione fabbrica che possiamo chiamare `make_closure()` (o `power()` nel codice dell'editor).

Analizziamo come lavorano le due diverse istanze create:

* **La prima closure** ottenuta da `make_closure()` definisce uno strumento che eleva al quadrato ($x^2$) l'argomento passato;
* **La seconda closure** è configurata invece per elevare al cubo ($x^3$) lo stesso argomento.

Sotto il cofano, Python mantiene separate le aree di memoria per i due ambienti estratti: quando invochi la prima istanza, questa ricorderà che l'esponente congelato è `2`, mentre la seconda saprà in modo indipendente che il suo esponente è `3`.